# Universidad de Buenos Aires - Maestria en Inteligencia Artificial - 2026

Entrenamiento de Razonamiento con GRPO + Phi-3 Mini

> **¿Qué se va a hacer en este cuaderno?** Se entrenara un modelo de lenguaje (Phi-3 Mini, de Microsoft) para que razone mejor usando un algoritmo llamado **GRPO** — usando GPU gratuita de Colab.

---

## Mapa del notebook

| Sección | Qué hace |
|---|---|
| 0 | Conectar Google Drive para guardar checkpoints |
| 1 | Instalar bibliotecas |
| 2 | Cargar el modelo Phi-3 Mini |
| 3 | Preparar el dataset de razonamiento |
| 4 | Definir las recompensas (para GRPO) |
| 5 | Configurar y lanzar el entrenamiento |
| 6 | Probar el modelo entrenado |

---

## ¿Qué es GRPO?

GRPO = **Group Relative Policy Optimization**

La idea central: en vez de usar un modelo separado para juzgar respuestas (como hace PPO), **el modelo se compara consigo mismo**.

```
Pregunta → Generar N respuestas → Puntuar cada una → Actualizar el modelo
                                          ↑
                               recompensas relativas al grupo
```

Así se puede enseñar al modelo a usar `<think>...</think>` antes de responder, como hace DeepSeek-R1.

**Paper original**: [DeepSeekMath (2024)](https://arxiv.org/abs/2402.03300)

---
## Sección 0: Conectar Google Drive

Se guardaran los checkpoints en Drive para no perder el progreso si Colab se desconecta.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Carpeta donde guardaremos todo
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/grpo_phi3_checkpoints'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f'✅ Drive montado. Checkpoints se guardarán en: {DRIVE_OUTPUT_DIR}')

Mounted at /content/drive
✅ Drive montado. Checkpoints se guardarán en: /content/drive/MyDrive/grpo_phi3_checkpoints


---
## Sección 1: Instalar librerías

Se usa:
- **`trl 1.4`**: librería de Hugging Face para entrenamiento con refuerzo (incluye GRPO)
- **`unsloth`**: acelera el entrenamiento en T4 con menos VRAM
- **`peft`**: para LoRA (se entrena solo una fracción de los parámetros)

In [2]:
%%capture
# Instalar sin output verbose (más limpio)
!pip install unsloth trl peft accelerate bitsandbytes datasets --quiet

# Verificar GPU disponible
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('🖥️ GPU disponible:', result.stdout.strip())

---
## Sección 2: Cargar Phi-3 Mini con LoRA

### ¿Por qué LoRA?
Phi-3 Mini tiene **3.8 mil millones de parámetros**. Entrenarlos todos requeriría ~28 GB de VRAM. Con LoRA se entrenan matrices pequeñas adicionales (adaptadores) — solo ~1-2% de parámetros — y la T4 gratuita puede manejarlo.

```
Parámetros originales (congelados): W
Adaptador LoRA:                     W' = W + α(A × B)   ← solo A y B se entrenan
```

In [3]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024  # Contexto máximo. Más largo = más VRAM.
LORA_RANK = 16         # Rango de LoRA. Más alto = más capacidad, más VRAM.

print('⏳ Cargando Phi-3 Mini (puede tardar 2-3 minutos la primera vez)...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-3-mini-4k-instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # Cuantización 4-bit: divide el uso de VRAM a la mitad
    fast_inference=False, # True requiere vLLM (no disponible en Colab gratuito)
)

# Agregar adaptadores LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],  # Capas de atención + FFN
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing='unsloth',  # Ahorra VRAM a costa de un poco de velocidad
    random_state=42,
)

print(f'✅ Modelo cargado con LoRA rank={LORA_RANK}')
print(f'   Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'   Total parámetros:       {sum(p.numel() for p in model.parameters()):,}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Cargando Phi-3 Mini (puede tardar 2-3 minutos la primera vez)...
==((====))==  Unsloth 2026.5.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Unsloth 2026.5.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Modelo cargado con LoRA rank=16
   Parámetros entrenables: 29,884,416
   Total parámetros:       2,039,024,640


---
## Sección 3: Preparar el dataset

Se usa **GSM8K** (Grade School Math 8K): 8500 problemas matemáticos de nivel escolar con soluciones paso a paso.

**¿Por qué matemáticas?** Porque tienen respuestas verificables objetivamente — perfectas para definir recompensas en GRPO.

El formato que se le pedira al modelo:
```
<think>
Aquí el modelo razona paso a paso...
</think>
La respuesta final es: 42
```

In [4]:
from datasets import load_dataset
import re

# Prompt del sistema: le decimos al modelo cómo debe comportarse
SYSTEM_PROMPT = """Eres un asistente de matemáticas que razona cuidadosamente.
SIEMPRE usa el siguiente formato:
<think>
Aquí vas paso a paso por el problema, sin apuro.
</think>
La respuesta final es: [número]"""

def formatear_ejemplo(ejemplo):
    """Convierte un ejemplo de GSM8K al formato de chat de Phi-3."""
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': ejemplo['question']}
        ],
        'answer': ejemplo['answer'],  # Guardamos la respuesta para calcular recompensas
    }

def extraer_numero_final(texto):
    """Extrae el número final de la respuesta del dataset (formato: '#### 42')."""
    match = re.search(r'####\s*([\d,\.]+)', texto)
    if match:
        return match.group(1).replace(',', '')
    return None

# Cargar GSM8K
print('📥 Descargando GSM8K...')
dataset = load_dataset('openai/gsm8k', 'main', split='train')
dataset = dataset.map(formatear_ejemplo)

# Usar un subconjunto para que quepa en tiempo de Colab gratuito
N_EJEMPLOS = 500
dataset = dataset.select(range(N_EJEMPLOS))

print(f'✅ Dataset listo: {len(dataset)} ejemplos')
print(f'\n📖 Ejemplo #0:')
print(f'  Pregunta: {dataset[0]["prompt"][1]["content"][:100]}...')
print(f'  Respuesta esperada: {extraer_numero_final(dataset[0]["answer"])}')

📥 Descargando GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

✅ Dataset listo: 500 ejemplos

📖 Ejemplo #0:
  Pregunta: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
  Respuesta esperada: 72


---
## Sección 4: Definir las funciones de recompensa

Para el agente RL **GRPO**.

Se usan tres recompensas combinadas:

| Recompensa | ¿Qué premia? | Valor |
|---|---|---|
| `recompensa_correctitud` | La respuesta final es correcta | 2.0 ó 0.0 |
| `recompensa_formato` | Usa `<think>...</think>` correctamente | 0.5 ó 0.0 |
| `recompensa_longitud_think` | El bloque think tiene longitud razonable | 0.0 a 0.3 |

In [5]:
import re

def extraer_respuesta_modelo(texto):
    """Extrae lo que el modelo dice después de 'La respuesta final es:'."""
    match = re.search(r'La respuesta final es:\s*([\d,\.]+)', texto)
    if match:
        return match.group(1).replace(',', '').strip()
    # Intentar extraer cualquier número al final si el formato no es perfecto
    numeros = re.findall(r'[\d]+\.?[\d]*', texto)
    return numeros[-1] if numeros else None

def recompensa_correctitud(completions, prompts, answer, **kwargs):
    """
    Recompensa principal: ¿es correcta la respuesta numérica?

    'completions' es una lista de respuestas generadas por el modelo.
    'answer' es la respuesta correcta del dataset.
    GRPO pasa estos en batch.
    """
    rewards = []
    for completion, correct in zip(completions, answer):
        texto = completion[0]['content']  # El texto generado
        respuesta_modelo = extraer_respuesta_modelo(texto)
        respuesta_correcta = extraer_numero_final(correct)

        if respuesta_modelo and respuesta_correcta:
            # Comparar como float para manejar decimales
            try:
                es_correcta = abs(float(respuesta_modelo) - float(respuesta_correcta)) < 0.01
                rewards.append(2.0 if es_correcta else 0.0)
            except ValueError:
                rewards.append(0.0)
        else:
            rewards.append(0.0)
    return rewards

def recompensa_formato(completions, **kwargs):
    """
    Recompensa de formato: ¿usa <think>...</think> correctamente?
    """
    rewards = []
    for completion in completions:
        texto = completion[0]['content']
        tiene_think = '<think>' in texto and '</think>' in texto
        tiene_respuesta = 'La respuesta final es:' in texto
        rewards.append(0.5 if (tiene_think and tiene_respuesta) else 0.0)
    return rewards

def recompensa_longitud_think(completions, **kwargs):
    """
    Recompensa de longitud: penaliza bloques <think> muy cortos (poco razonamiento)
    y muy largos (verbosidad excesiva). Rango óptimo: 100-500 chars.
    """
    rewards = []
    for completion in completions:
        texto = completion[0]['content']
        match = re.search(r'<think>(.*?)</think>', texto, re.DOTALL)
        if match:
            longitud = len(match.group(1).strip())
            # Recompensa máxima entre 100 y 500 caracteres
            if 100 <= longitud <= 500:
                rewards.append(0.3)
            elif longitud < 100:
                rewards.append(0.1)  # Muy corto: poco razonamiento
            else:
                rewards.append(0.1)  # Muy largo: verboso
        else:
            rewards.append(0.0)
    return rewards

print('✅ Funciones de recompensa definidas.')
print('   Recompensa máxima posible por respuesta: 2.0 + 0.5 + 0.3 = 2.8')

✅ Funciones de recompensa definidas.
   Recompensa máxima posible por respuesta: 2.0 + 0.5 + 0.3 = 2.8


---
## Sección 5: Configurar y lanzar el entrenamiento GRPO

### Hiperparámetros clave explicados:

| Parámetro | Valor | Significado |
|---|---|---|
| `num_generations` | 4 | Por cada pregunta, genera 4 respuestas y las compara entre sí |
| `max_new_tokens` | 512 | Longitud máxima de cada respuesta generada |
| `learning_rate` | 5e-6 | Paso de aprendizaje pequeño (RL es inestable si es grande) |
| `kl_coeff` | 0.1 | Penalización por alejarse demasiado del modelo original |
| `per_device_train_batch_size` | 1 | Batch size pequeño para caber en T4 (15GB VRAM) |

In [6]:
from trl import GRPOConfig, GRPOTrainer

# ──────────────────────────────────────────────────────────
# CONFIGURACIÓN DEL ENTRENAMIENTO
# ──────────────────────────────────────────────────────────
training_args = GRPOConfig(
    # --- Rutas ---
    output_dir=DRIVE_OUTPUT_DIR,           # Guardar en Drive

    # --- Generación de respuestas ---
    num_generations=4,                     # Respuestas por pregunta para comparar

    # --- Batches y pasos ---
    per_device_train_batch_size=1,         # 1 pregunta a la vez (T4 es limitada)
    gradient_accumulation_steps=4,         # Acumula gradientes de 4 pasos antes de actualizar
    max_steps=200,                         # Pasos totales (~30-40 min en T4)

    # --- Optimizador ---
    learning_rate=5e-6,
    lr_scheduler_type='cosine',            # Reducir LR gradualmente
    warmup_steps=10,                       # Arranque suave
    optim='adamw_8bit',                    # AdamW en 8-bit para ahorrar VRAM

    # --- KL Divergence (regularización) ---
    #kl_coeff=0.1,                          # No te alejes demasiado del modelo original
    beta=0.1,

    # --- Checkpoints en Drive ---
    save_steps=50,                         # Guardar cada 50 pasos
    save_total_limit=3,                    # Mantener solo los 3 checkpoints más recientes
    save_strategy='steps',

    # --- Logging ---
    logging_steps=10,
    report_to='none',                      # Sin W&B ni TensorBoard por simplicidad

    # --- Precisión ---
    fp16=True,                             # Float16 para T4
)

print('✅ Configuración lista.')
print(f'   Steps totales: {training_args.max_steps}')
print(f'   Checkpoints cada: {training_args.save_steps} steps')
print(f'   Guardando en: {training_args.output_dir}')

✅ Configuración lista.
   Steps totales: 200
   Checkpoints cada: 50 steps
   Guardando en: /content/drive/MyDrive/grpo_phi3_checkpoints


In [7]:
# Inicializar el Trainer de GRPO
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        recompensa_correctitud,      # Más importante: respuesta correcta
        recompensa_formato,          # Usa el formato <think>
        recompensa_longitud_think,   # Razona con longitud adecuada
    ],
    generation_config={
        'max_new_tokens': 512,
        'temperature': 0.7,
        'do_sample': True,
    },
    args=training_args,
    train_dataset=dataset,
)

print('✅ Trainer inicializado con 3 funciones de recompensa.')
print('\n⚡ Iniciando entrenamiento GRPO...')
print('   (Los primeros pasos son lentos mientras se calienta la GPU)')
print('   Tip: Observa cómo la recompensa promedio sube con los pasos.\n')

trainer.train()

✅ Trainer inicializado con 3 funciones de recompensa.

⚡ Iniciando entrenamiento GRPO...
   (Los primeros pasos son lentos mientras se calienta la GPU)
   Tip: Observa cómo la recompensa promedio sube con los pasos.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureW

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / recompensa_correctitud / mean,rewards / recompensa_correctitud / std,rewards / recompensa_formato / mean,rewards / recompensa_formato / std,rewards / recompensa_longitud_think / mean,rewards / recompensa_longitud_think / std
10,0.000001,1.432500,0.752656,219.475000,177.500000,249.300000,0.400000,196.508334,177.500000,214.500000,0.000014,1.300000,0.630940,0.075000,0.128868,0.057500,0.056962
20,0.000002,1.137500,1.108308,220.250000,183.300000,246.700000,0.450000,181.008334,157.700000,196.500000,0.000020,1.050000,1.077350,0.037500,0.053868,0.050000,0.054142
30,0.000002,1.237500,0.736993,221.075000,176.800000,253.400000,0.375000,180.858336,151.200000,206.200000,0.000017,1.150000,0.646410,0.037500,0.075000,0.050000,0.069916
40,0.000004,1.250000,0.762315,212.800000,180.700000,249.600000,0.400000,149.691670,129.500000,167.100000,0.000036,1.150000,0.730940,0.050000,0.078868,0.050000,0.070415
50,0.000003,1.277500,0.832261,215.450000,179.000000,242.000000,0.450000,168.125000,153.400000,182.900000,0.000030,1.150000,0.761880,0.050000,0.078868,0.077500,0.096463
60,0.000003,1.115000,0.864350,223.575000,190.900000,256.000000,0.475000,158.450002,139.700000,175.800000,0.000031,0.950000,0.815470,0.087500,0.132735,0.077500,0.086725
70,0.000005,1.147500,0.653067,202.775000,185.400000,221.200000,0.425000,123.575000,108.600000,141.500000,0.000053,1.100000,0.630940,0.000000,0.000000,0.047500,0.060689
80,0.000007,1.060000,0.654512,202.375000,170.000000,234.100000,0.350000,168.158334,144.400000,194.500000,0.000069,1.050000,0.646410,0.000000,0.000000,0.010000,0.015774
90,0.000009,0.795000,0.813499,229.975000,190.900000,256.000000,0.500000,187.233334,165.300000,207.300000,0.000094,0.750000,0.815470,0.025000,0.050000,0.020000,0.027583
100,0.000011,0.945000,0.654008,221.725000,175.900000,249.500000,0.400000,169.408334,150.300000,183.800000,0.000113,0.850000,0.615470,0.037500,0.075000,0.057500,0.103284


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=25

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/grpo_phi3_checkpoints/checkpoint-50.
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the

TrainOutput(global_step=200, training_loss=1.3570405644713902e-05, metrics={'train_runtime': 7494.771, 'train_samples_per_second': 0.107, 'train_steps_per_second': 0.027, 'total_flos': 0.0, 'train_loss': 1.3570405644713902e-05})

In [8]:
# Guardar el modelo final en Drive
MODELO_FINAL_DIR = f'{DRIVE_OUTPUT_DIR}/modelo_final'
os.makedirs(MODELO_FINAL_DIR, exist_ok=True)

# Guardar pesos LoRA (solo los adaptadores, no el modelo completo)
model.save_pretrained(MODELO_FINAL_DIR)
tokenizer.save_pretrained(MODELO_FINAL_DIR)

print(f'✅ Modelo guardado en: {MODELO_FINAL_DIR}')
print('   (Solo se guardan los pesos LoRA, ~50-100 MB, no los 3.8B del modelo base)')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/grpo_phi3_checkpoints/modelo_final/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/grpo_phi3_checkpoints/modelo_final.


✅ Modelo guardado en: /content/drive/MyDrive/grpo_phi3_checkpoints/modelo_final
   (Solo se guardan los pesos LoRA, ~50-100 MB, no los 3.8B del modelo base)


---
## Sección 6: Probar el modelo entrenado

Aqui comprueba si el modelo aprendió a razonar con el formato `<think>`.

In [9]:
from unsloth import FastLanguageModel

# Cambiar a modo inferencia (más rápido)
model.eval()  # Modo evaluación (fast_inference=False, sin vLLM)

def generar_respuesta(pregunta, max_tokens=512):
    """Genera una respuesta usando el modelo entrenado."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pregunta}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_tokens,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decodificar solo los tokens nuevos
    respuesta = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return respuesta

# ──────────────────────────────────────────────────────────
# PREGUNTAS DE PRUEBA
# ──────────────────────────────────────────────────────────
preguntas_prueba = [
    'Si tengo 24 manzanas y doy 1/3 a mi hermana y luego como 4, ¿cuántas me quedan?',
    'Un tren viaja a 60 km/h. ¿Cuánto tarda en recorrer 180 km?',
    'María tiene el doble de edad que Juan. Juan tiene 15 años. ¿Cuántos años tendrá María en 5 años?',
]

for i, pregunta in enumerate(preguntas_prueba, 1):
    print(f'\n{'='*60}')
    print(f'PREGUNTA {i}: {pregunta}')
    print('='*60)
    respuesta = generar_respuesta(pregunta)
    print(respuesta)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PREGUNTA 1: Si tengo 24 manzanas y doy 1/3 a mi hermana y luego como 4, ¿cuántas me quedan?


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Para resolver este problema, primero debemos calcular cuántas manzanas se le dan a la hermana. Dado que se nos informa que 1/3 de las manzanas se dan, multiplicamos el total de manzanas, que es 24, por 1/3:

24 * (1/3) = 8 manzanas.

Ahora, restamos las 8 manzanas dadas a la hermana del total original:

24 - 8 = 16 manzanas.

Luego, restamos las 4 manzanas que se comen:

16 - 4 = 12 manzanas.

Por lo tanto, después de dar a la hermana 1/3 de las manzanas y comer 4, se nos deja con 12 manzanas.
</think>

La respuesta final es: 12

PREGUNTA 2: Un tren viaja a 60 km/h. ¿Cuánto tarda en recorrer 180 km?


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Para resolver este problema, primero identificamos que queremos encontrar el tiempo que tarda el tren en recorrer una cierta distancia.

La fórmula que relaciona velocidad, distancia y tiempo es:

\[ \text{Velocidad (v)} = \frac{\text{Distancia (d)}}{\textzeichnung (t)} \]

Por lo tanto, para encontrar el tiempo (t), reorganizamos la fórmula para obtener:

\[ t = \frac{d}{v} \]

Ahora, sustituimos los valores dados:

\[ t = \frac{180 \text{ km}}{60 \text{ km/h}} \]

Al realizar la operación, obtenemos:

\[ t = 3 \text{ horas} \]

La respuesta final es: 3

PREGUNTA 3: María tiene el doble de edad que Juan. Juan tiene 15 años. ¿Cuántos años tendrá María en 5 años?
<think>
Para resolver este problema, primero identificamos la información clave:
1. María tiene el doble de edad que Juan.
2. Juan tiene 15 años.
3. Necesitamos saber la edad de María en 5 años.


Primero, calculamos la edad actual de María multiplicando la edad de Juan por 2, ya que ella tiene el doble de edad que Juan

In [10]:
# ──────────────────────────────────────────────────────────
# CELDA INTERACTIVA: prueba tu propia pregunta
# ──────────────────────────────────────────────────────────
tu_pregunta = 'Si un producto cuesta $45 y tiene un descuento del 20%, ¿cuánto pagas?'  # @param {type:"string"}

print('Tu pregunta:', tu_pregunta)
print('\nRespuesta del modelo:')
print('-' * 40)
respuesta = generar_respuesta(tu_pregunta)
print(respuesta)

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Tu pregunta: Si un producto cuesta $45 y tiene un descuento del 20%, ¿cuánto pagas?

Respuesta del modelo:
----------------------------------------
<think>
Primero, se calcula el monto del descuento aplicando el porcentaje al precio original del producto.
Descuento = Precio original × Porcentaje de descuento
Descuento = $45 × 20%
Descuento = $45 × 0ños
Descuento = $9
Ahora se resta el descuento al precio original para obtener el precio final.
Precio final = Precio original - Descuento
Precio final = $45 - $9
Precio final = $36
Por lo tanto, después de aplicar un descuento del 20%, pagas $36 por el producto.
</think>
La respuesta final es: 36


---
## Análisis: ¿Qué aprendió el modelo?

Para responder eso se corre una evaluación rápida en el conjunto de validación de GSM8K.

In [11]:
from tqdm.auto import tqdm

# Cargar 50 ejemplos de validación
val_dataset = load_dataset('openai/gsm8k', 'main', split='test').select(range(50))

correctas = 0
con_formato = 0
total = len(val_dataset)

print(f'📊 Evaluando en {total} ejemplos de validación...')

for ejemplo in tqdm(val_dataset):
    respuesta = generar_respuesta(ejemplo['question'], max_tokens=400)

    # Verificar formato
    if '<think>' in respuesta and '</think>' in respuesta:
        con_formato += 1

    # Verificar correctitud
    respuesta_modelo = extraer_respuesta_modelo(respuesta)
    respuesta_correcta = extraer_numero_final(ejemplo['answer'])

    if respuesta_modelo and respuesta_correcta:
        try:
            if abs(float(respuesta_modelo) - float(respuesta_correcta)) < 0.01:
                correctas += 1
        except ValueError:
            pass

print(f'\n📈 RESULTADOS:')
print(f'   Accuracy:         {correctas}/{total} = {correctas/total*100:.1f}%')
print(f'   Usa formato think: {con_formato}/{total} = {con_formato/total*100:.1f}%')
print(f'\n💡 Phi-3 Mini sin GRPO tiene ~55-60% en GSM8K.')
print(f'   Si tu modelo supera eso, ¡el entrenamiento funcionó!')

📊 Evaluando en 50 ejemplos de validación...


  0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


📈 RESULTADOS:
   Accuracy:         32/50 = 64.0%
   Usa formato think: 22/50 = 44.0%

💡 Phi-3 Mini sin GRPO tiene ~55-60% en GSM8K.
   Si tu modelo supera eso, ¡el entrenamiento funcionó!


---
## Resumen

### ¿Qué se hizo?

1. **Se elige** un modelo base (Phi-3 Mini) que ya sabe generar texto
2. **Se definen recompensas**: qué significa una respuesta buena (correcta, bien formateada, razonada)
3. **GRPO generó** múltiples respuestas por pregunta y las comparó entre sí
4. **El modelo aprendió** a preferir las respuestas que maximizan las recompensas

### La diferencia clave vs SFT (Supervised Fine-Tuning)

| | SFT | GRPO |
|---|---|---|
| Aprende de | Ejemplos correctos dados | Sus propias respuestas evaluadas |
| Necesita | Datos de alta calidad | Solo una función de evaluación |
| Emergencia | No | Sí: puede descubrir estrategias nuevas |
| Escala | Limitado por datos | Limitado por compute |

### ¿Para qué más puedes usar GRPO?

- **Código**: recompensar por tests que pasan
- **Lógica**: recompensar por proofs válidos
- **Escritura**: recompensar por preferencias humanas
- **Cualquier tarea** donde se pueda definir "qué es correcto"

### Lecturas recomendadas

- [DeepSeekMath paper](https://arxiv.org/abs/2402.03300) — GRPO original
- [TRL docs - GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer)
- [LoRA paper](https://arxiv.org/abs/2106.09685) — por qué funciona LoRA
- [Phi-3 Technical Report](https://arxiv.org/abs/2404.14219)

---
*Notebook diseñado para correr en Google Colab T4 gratuito. Tiempo estimado: 45-60 minutos.*